# Import Libraries

In [1]:
import torch
from torch.utils.data import Subset, Dataset, DataLoader
from tqdm import tqdm
import multiprocessing as mp

from models.gpt import GPT
from dataset.dataset import OthelloDataset
from game.othello import GameBoard, Piece
from models.linear_probe import LinearProbe

# Load  a target model

In [10]:
def load_checkpoint(model, checkpoint):
    checkpoint = torch.load("../checkpoints/" + checkpoint)
    model.load_state_dict(checkpoint)

model = GPT()
model_name = "gpt_batch_2400_loss_2.0949_pass_rate_96.3701"
load_checkpoint(model, model_name + ".pt")

# Create dataset

In [ ]:
NUM_OF_GAMES_FOR_PROBE_TRAINING = 21_000
NUM_OF_GAMES_USED_TO_EVALUATE_GPT = 1000
MAX_GAME_LENGTH = 60
BOARD_SIZE = 64
PADDING_TOKEN = 0

test_dataset = OthelloDataset(train=False, path="../dataset/test_dataset.pt")
probe_dataset = Subset(test_dataset, range(NUM_OF_GAMES_USED_TO_EVALUATE_GPT, NUM_OF_GAMES_FOR_PROBE_TRAINING + NUM_OF_GAMES_USED_TO_EVALUATE_GPT))

def convert_token_to_position(token):
    flat_index = token - 1

    if flat_index >= 33:
        flat_index += 4

    elif flat_index >= 27:
        flat_index += 2

    row = flat_index // 8
    col = flat_index % 8

    return GameBoard.index_to_position((row, col))


# Calculate total number of sequences
num_of_sequences = 0
temp = torch.empty((NUM_OF_GAMES_FOR_PROBE_TRAINING, MAX_GAME_LENGTH))
for i in range(NUM_OF_GAMES_FOR_PROBE_TRAINING):
    temp[i] = probe_dataset[i][0]
num_of_sequences = temp.ne(0).sum().item()

# Create X and Y
X = torch.full((num_of_sequences, MAX_GAME_LENGTH), PADDING_TOKEN)
Y = torch.full((num_of_sequences, 64, 3), 0)
idx = 0
for i in range(NUM_OF_GAMES_FOR_PROBE_TRAINING):
    x = probe_dataset[i][0]
    for j in range(MAX_GAME_LENGTH):
        if x[j] == PADDING_TOKEN:
            break

        X[idx, :j + 1] = x[:j + 1]
        idx += 1

def process_sequence(args):
    i, x_seq = args
    x_filtered = x_seq[x_seq.ne(PADDING_TOKEN)]
    list_of_positions = [convert_token_to_position(token.item()) for token in x_filtered]

    game_board = GameBoard()
    current_player = Piece.BLACK

    for position in list_of_positions:
        game_board.add_piece(current_player, position)

        current_player = Piece.WHITE if current_player == Piece.BLACK else Piece.BLACK
        if not game_board.get_legal_moves(current_player):
            opponent = Piece.WHITE if current_player == Piece.BLACK else Piece.BLACK
            if game_board.get_legal_moves(opponent):
                current_player = opponent

    game_board = game_board.get_board().flatten()
    y = torch.zeros((BOARD_SIZE, 3))
    opponent = Piece.WHITE if current_player == Piece.BLACK else Piece.BLACK

    mine_mask = (game_board == current_player)
    yours_mask = (game_board == opponent)
    empty_mask = ~(mine_mask | yours_mask)

    y[mine_mask, 0] = 1.0
    y[yours_mask, 1] = 1.0
    y[empty_mask, 2] = 1.0

    return i, y

if __name__ == '__main__':
    tasks = [(i, X[i]) for i in range(num_of_sequences)]
    with mp.Pool(processes=mp.cpu_count()) as pool:
        for i, y in tqdm(pool.imap_unordered(process_sequence, tasks, chunksize=250),
                         total=num_of_sequences,
                         desc="Generating Y labels"):
            Y[i] = y

    torch.save(X, "probes_dataset/sequences.pt")
    torch.save(Y, "probes_dataset/game_boards.pt")

# Load dataset

In [19]:
from torch.utils.data import Dataset
import torch
from tqdm import tqdm

layer_num = 1

class ProbeDataset(Dataset):
    def __init__(self, train=True, model=None, layer_num=layer_num, split_ratio=0.9):
        X = torch.load("probes_dataset/sequences.pt")
        Y = torch.load("probes_dataset/game_boards.pt")

        split_idx = int(len(X) * split_ratio)

        if train:
            self.X = X[:split_idx]
            self.Y = Y[:split_idx]
        else:
            self.X = X[split_idx:]
            self.Y = Y[split_idx:]

        self.model = model
        self.layer_num = layer_num

        if self.model is not None:
            self.__precompute_hidden_states()

    def __precompute_hidden_states(self, batch_size=1024):
        self.model.eval()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(device)

        self.hidden_states = torch.empty(self.X.size(dim=0), self.model.d_model, device="cpu")

        with torch.no_grad():
            for i in tqdm(range(0, len(self.X), batch_size), desc="Precomputing hidden states"):
                batch_x = self.X[i : i + batch_size].to(device)

                hidden_states_batch = self.model.get_hidden_state(batch_x, layer_num=self.layer_num)

                lengths = (batch_x != 0).sum(dim=1)

                last_move_indices = lengths - 1

                bs = hidden_states_batch.size(0)
                batch_indices = torch.arange(bs, device=device)

                correct_hidden_states = hidden_states_batch[batch_indices, last_move_indices, :]

                self.hidden_states[i : i + bs] = correct_hidden_states.cpu()

    def __len__(self):
        return self.hidden_states.size(dim=0)

    def __getitem__(self, idx):
        return self.hidden_states[idx], self.Y[idx]

In [20]:
dat = ProbeDataset(model=model)

Precomputing hidden states: 100%|██████████| 1108/1108 [02:27<00:00,  7.53it/s]


In [21]:
val = ProbeDataset(train=False, model=model)

Precomputing hidden states: 100%|██████████| 124/124 [00:16<00:00,  7.56it/s]


# Probes training

In [22]:
train_loader = DataLoader(dat, batch_size=1024, shuffle=True)

In [ ]:
probe = LinearProbe(model.d_model, 64*3)

def train(num_epoch=100, probe=None, train_loader=None, device="cuda"):
    probe.to(device)
    optimizer = probe.get_optimizer()
    loss_fn = probe.get_loss_fn()

    for epoch in range(num_epoch):
        probe.train()
        full_loss = 0
        correct_cells = 0
        total_cells = 0

        for x_hidden, y_labels in tqdm(train_loader):
            x = x_hidden.to(device)
            y = y_labels.to(device).float()

            logits = probe(x)

            logits_flat = logits.view(-1, 3)
            targets_flat = y.view(-1, 3)

            loss = loss_fn(logits_flat, targets_flat)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                preds = torch.argmax(logits_flat, dim=1)
                actuals = torch.argmax(targets_flat, dim=1)

                correct_cells += (preds == actuals).sum().item()
                total_cells += actuals.size(0)

            full_loss += loss.item()

        avg_loss = full_loss / len(train_loader)
        accuracy = (correct_cells / total_cells) * 100

        print(f"Epoch {epoch+1:03d}/{num_epoch} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

    return probe

torch.save(train(12, probe, train_loader).state_dict(), f"probes/linear_probe_layer_{layer_num}_{model_name}.pt")

# Probe evaluation

In [17]:
val_loader = DataLoader(val, batch_size=1024, shuffle=True)

In [18]:
def validate(probe=None, val_loader=None, device="cuda"):
    probe.to(device)
    probe.eval()

    loss_fn = probe.get_loss_fn()

    full_loss = 0
    correct_cells = 0
    total_cells = 0

    with torch.no_grad():
        for x_hidden, y_labels in tqdm(val_loader):
            x = x_hidden.to(device)
            y = y_labels.to(device).float()

            logits = probe(x)

            logits_flat = logits.view(-1, 3)
            targets_flat = y.view(-1, 3)

            loss = loss_fn(logits_flat, targets_flat)

            preds = torch.argmax(logits_flat, dim=1)
            actuals = torch.argmax(targets_flat, dim=1)

            correct_cells += (preds == actuals).sum().item()
            total_cells += actuals.size(0)

            full_loss += loss.item()

    avg_loss = full_loss / len(val_loader)
    accuracy = (correct_cells / total_cells) * 100

    print(f"[VAL] Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

    return avg_loss, accuracy

validate(probe, val_loader)


100%|██████████| 124/124 [00:00<00:00, 215.92it/s]

[VAL] Loss: 0.7267 | Accuracy: 64.34%


(0.7266919305247646, 64.33655120948215)

In [ ]:


"""
Model: gpt_batch_2400_loss_2.0949_pass_rate_96.3701
Layer 0: Loss: 0.7267 | Accuracy: 64.34%
Layer 1: Loss: 0.2213 | Accuracy: 89.13%
Layer 2: Loss: 0.1521 | Accuracy: 93.13%
Layer 3: Loss: 0.1105 | Accuracy: 95.29%
Layer 4: Loss: 0.1078 | Accuracy: 95.48%

Model: gpt_baseline
Layer 0: Loss: 0.8687 | Accuracy: 58.78%
Layer 1: Loss: 0.3492 | Accuracy: 77.12%
Layer 2: Loss: 0.3507 | Accuracy: 77.06%
Layer 3: Loss: 0.3532 | Accuracy: 76.98%
Layer 4: Loss: 0.3561 | Accuracy: 76.87%
"""